# GACC Fire Weather — AGOL data pipeline

Feeds the **GACC Fire Wx** Experience Builder app. Each run:

1. pulls D-30 … D+7 NFDR daily summaries from FEMS for every key RAWS station,
2. averages to PSA level and compares against the 2005–2024 climo baseline,
3. updates three items in the *GACC Fire Weather* folder (created on the first run):
   - **GACC Fire Wx – PSA Current** — PSA polygons with today's values, alerts and 7-day peaks
   - **GACC Fire Wx – PSA Daily** — PSA × date table for charts
   - **GACC Fire Wx – Map** — web map styled by alert level (created once; your edits are kept)

### One-time setup
1. In *Files* (`/arcgis/home`), create **`fems_credentials.env`** containing:
   ```
   FEMS_API_KEY=your_key
   FEMS_USERNAME=your.email@usda.gov
   ```
   `/arcgis/home` is visible only to you. Don't put the key in this notebook.
2. Run all cells once by hand and check the log for warnings (PSAs with no boundary / no climo).
3. **Tasks → Create scheduled task**: daily, twice (≈ 06:30 and 16:30 Pacific) so the afternoon NWS
   NFDR forecast is picked up. Runtime ≈ 3–6 min (≈ 875 stations × 1 request).


In [ ]:
# ── Settings ────────────────────────────────────────────────────────────
REPO      = 'shelikes2run/GACC-fire-dashboard'
REF       = 'agol-experience-builder'  # switch to 'main' after PR #1 is merged
WORK_DIR  = '/arcgis/home/gacc_fire_wx'
CRED_FILE = '/arcgis/home/fems_credentials.env'
GACCS     = None                       # e.g. ['OSCC'] for a quick test run


In [ ]:
# ── Pull current code, PSA config and climo baseline from GitHub ─────────
import os, requests, pathlib
FILES = ['fems_fetcher.py', 'gacc_config.py', 'gacc_climo_baseline.json', 'agol/agol_pipeline.py']
root = pathlib.Path(WORK_DIR); (root / 'agol').mkdir(parents=True, exist_ok=True)
for f in FILES:
    r = requests.get(f'https://raw.githubusercontent.com/{REPO}/{REF}/{f}', timeout=60)
    r.raise_for_status()
    (root / f).write_bytes(r.content)
    print(f'{f:32s} {len(r.content):>9,d} bytes')


In [ ]:
# ── Run ─────────────────────────────────────────────────────────────────
import sys, logging, importlib
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(message)s', datefmt='%H:%M:%S', force=True)
sys.path.insert(0, str(root / 'agol'))
import agol_pipeline as ap; importlib.reload(ap)
ap.read_cred_file(CRED_FILE)
result = ap.run(repo_dir=root, gaccs=GACCS)


### Notes
- First run creates the items **private**. Share the three items and the Experience with your group when ready.
- The PSA Current layer keeps its geometry after the first run (only attributes are updated). If NIFC
  redraws PSA boundaries, delete that item and re-run to rebuild it — then repoint the web map layers.
- Fuel-moisture alerts are intentionally off: the baseline percentiles for FM are wet-side, so a
  ≥ 97th test would flag wet days. Values and thresholds still publish for charts.
